# Pilot

The pilot's runs, one phase at a time (`PHASE`), five per cell and condition at p_inter = 0, 0.125, …, 0.5 at
H = 100 ([`schedule.md`](schedule.md#pilot) § Pilot). They
test the code and give each cell's largest `prompt_eval_count` against `LLM_NUM_CTX`;
phase 2 starts as the runs do, community 0 on +1 and community 1 on −1, and
[`runs_analysis.ipynb`](runs_analysis.ipynb) pools it with the main runs at H.
Phase 1 starts on drawn poles, each with probability ½, and is never pooled.

## 1. Load

Every run directory of the phase copied back, phase 1's to `logs/pilot/phase1/`, phase 2's to `logs/pilot/phase2/`, one row each in `runs`: its cell,
condition, p_inter, the last round logged and its provenance. A run still going or
aborted has no round H + 1 (`metrics.is_complete`); `events` holds the complete runs'.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.precision", 3)  # every table at three decimals

CWD = Path.cwd().resolve()
REPO = next(p for p in [CWD, *CWD.parents] if (p / "simulation" / "config.py").is_file())
sys.path.insert(0, str(REPO / "analysis"))
import metrics  # noqa: E402

PHASE = 2   # 2: the polarized start (schedule.md § Pilot)
PILOT = REPO / "analysis" / "logs" / "pilot" / f"phase{PHASE}"
STG = "P" if PHASE == 1 else "P2"
PROVENANCE = ["git_commit", "model_digest", "ollama_version", "node"]

rows = []
for d in sorted(PILOT.glob(f"run_{STG}-*")):
    if not (d / "params.json").is_file():  # mid-copy
        continue
    p = metrics.read_params(d)
    rounds = [int(f.stem.removeprefix("round_")) for f in (d / "network_rounds").glob("round_*.json")]
    rows.append({"run": d.name, "topic": p["TOPIC_KEY"], "model": p["LLM_MODEL"],
                 "condition": p["CONDITION"], "p_inter": p["SBM_P_INTER"],
                 "last_round": max(rounds, default=None), "complete": metrics.is_complete(d),
                 **{k: p[k] for k in PROVENANCE}})
if not rows:
    raise SystemExit(f"no run_{STG}-* directory with params.json under {PILOT}")
runs = pd.DataFrame(rows).set_index("run")
events = {r: metrics.read_events(PILOT / r) for r in runs.index[runs.complete]}

print(runs.complete.sum(), "of", len(runs), "runs complete")
display(runs.pivot_table("last_round", ["topic", "model"], ["condition", "p_inter"], aggfunc="max")
        .astype("Int64"))
display(runs.groupby(["model", *PROVENANCE]).size().rename("runs"))


## 2. D over the rounds

The divergence D = |m₀ − m₁| / 2 at every snapshot, rounds 0 to H + 1 (the last is the final
declarations), averaged over each cell's complete runs, pooled over p_inter, each run's own curve faint behind the mean (one run per p_inter); by model (rows) and
topic (columns), one grid per condition, `full` on the left and `own` on the right
(`own` shows the agent's own turns alone, [`simulation/README.md`](../simulation/README.md#memory)
§ Memory), n runs per panel. The second figure zooms into the
first 20 rounds, where the start (round 0) gives way to the first exchanges.
A descriptive look at how the state moves, not a test.

In [ ]:
import itertools

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

COLOR = "#2a78d6"
MODELS, TOPICS = sorted(runs.model.unique()), sorted(runs.topic.unique())
CONDS = ("full", "own")
curves = {r: metrics.run_divergence(PILOT / r) for r in runs.index[runs.complete]}



def cond_grid(xlabel, ylabel, figsize=(16, 6)):
    """A model x topic grid per condition, `full` left and `own` right with a gap between, titled
    by topic and condition, labelled on the outer edges. Returns the figure, every axis and the panels
    as (ax, cond, model, topic)."""
    nt = len(TOPICS)
    fig, grid = plt.subplots(len(MODELS), 2 * nt + 1, figsize=figsize, sharex=True, sharey=True,
                             squeeze=False, gridspec_kw={"width_ratios": [1] * nt + [0.3] + [1] * nt})
    for ax in grid[:, nt]:  # the gap between the conditions
        ax.set_visible(False)
    panels = []
    for c, cond in enumerate(CONDS):
        axes = grid[:, c * (nt + 1):c * (nt + 1) + nt]
        for (i, model), (j, topic) in itertools.product(enumerate(MODELS), enumerate(TOPICS)):
            ax = axes[i, j]
            if i == 0:
                ax.set_title(topic, fontsize=10, pad=14)  # above n_label
            if j == 0 and c == 0:
                ax.set_ylabel(f"{model.split('-')[0]}\n{ylabel}", fontsize=9)
            if i == len(MODELS) - 1:
                ax.set_xlabel(xlabel)
            panels.append((ax, cond, model, topic))
        axes[0, nt // 2].annotate(f"condition {cond}", (0.5, 1), xycoords="axes fraction", xytext=(0, 30),
                                  textcoords="offset points", ha="center", va="bottom", fontsize=11,
                                  weight="bold")
    return fig, grid, panels


def n_label(ax, n):
    ax.text(1, 1.02, f"n = {n}" if n else "no runs yet", transform=ax.transAxes, ha="right", va="bottom",
            color="0.5", fontsize=8)


def plot_grid(lines, ylabel, title, ylim, first_rounds=None, community_legend=False):
    """`cond_grid`'s panels, each line the mean over the cell's complete runs, pooled over p_inter, each
    run's own line faint behind it. `lines(run)` gives that run's (series, linestyle) pairs;
    `first_rounds` keeps rounds 0 … that many."""
    fig, _, panels = cond_grid("round", ylabel)
    for ax, cond, model, topic in panels:
        cell = runs[(runs.model == model) & (runs.topic == topic) & (runs.condition == cond) & runs.complete]
        per_run = [lines(r) for r in cell.index]
        for k, (_, ls) in enumerate(per_run[0] if per_run else []):
            for pr in per_run:
                t = pr[k][0] if first_rounds is None else pr[k][0].loc[:first_rounds]
                ax.plot(t.index, t, color=COLOR, ls=ls, lw=0.8, alpha=0.25)
            s = pd.concat([pr[k][0] for pr in per_run], axis=1).mean(axis=1)
            s = s if first_rounds is None else s.loc[:first_rounds]
            ax.plot(s.index, s, color=COLOR, ls=ls, lw=1.6, marker=None if first_rounds is None else "o", ms=3)
        ax.set_ylim(*ylim)
        ax.grid(axis="y", color="0.92")
        ax.spines[["top", "right"]].set_visible(False)
        n_label(ax, len(per_run))
    if community_legend:
        fig.legend(handles=[Line2D([], [], color=COLOR, lw=1.6, ls=ls, label=f"community {k}")
                            for k, ls in enumerate(("-", "--"))], loc="center right", frameon=False, fontsize=8)
    window = "" if first_rounds is None else f", first {first_rounds} rounds"
    fig.suptitle(f"{title}, mean over p_inter (faint: each run){window}", fontsize=11)
    fig.tight_layout(rect=(0, 0, 0.93 if community_legend else 1, 0.95))
    plt.show()


## 3. Community means of the declared position

Each community's mean declared position y (each y −1 or +1) at every snapshot, averaged over each cell's complete runs, pooled over p_inter, on the
grid of § 2: solid for community 0, dashed for community 1. In phase 1 the labels 0 and 1 are arbitrary per
run, so the average shows the cell's common drift; two communities parting in opposite directions
in different runs cancel here and show in § 2's D instead. In phase 2 community 0 always starts at +1, so the
average shows whether the two hold their poles. A descriptive look, not a test.

In [ ]:
means = {r: metrics.run_community_means(PILOT / r) for r in runs.index[runs.complete]}

plot_grid(lambda r: [(means[r][0], "-"), (means[r][1], "--")], "mean position",
          "Community means of the declared position", (-1.05, 1.05), community_legend=True)

## 4. Dispersion of the declared positions

The variance of the declared poles over the whole network at every snapshot (`metrics.run_dispersion`),
the benchmark's measure, on the grid of § 2, averaged over each cell's complete runs, pooled over
p_inter: 0 at consensus, 1 at an even division. It ignores the communities, so it also shows a
division that runs across them. A descriptive look, not a test.

In [ ]:
dispersions = {r: metrics.run_dispersion(PILOT / r) for r in runs.index[runs.complete]}

plot_grid(lambda r: [(dispersions[r], "-")], "dispersion", "Dispersion of y", (-0.03, 1.03))

## 5. Distribution of the declared positions

The share of declarations on each pole y (−1 or +1), pooled
over each cell's complete runs and p_inter, on the grid of § 2. Every modelled declaration counts
(`metrics.run_declarations`: in the exchanges and at round H + 1); the starts, which stand
undeclared, do not. n is the declarations per cell. A descriptive look, not a test.

In [ ]:
POS = list(metrics.POSITIONS)


def window_mean(d, k):
    """The mean of entry field k (0: ŝ, 1: ô) over a declaration's window; NaN without one."""
    vals = [e[k] for e in d["window"] if e[k] is not None]
    return sum(vals) / len(vals) if vals else float("nan")


decls = pd.DataFrame([{"run": r, "agent": d["agent"], "y": d["y"], "s_hat": window_mean(d, 0), "o_hat": window_mean(d, 1)}
                      for r in runs.index[runs.complete] for d in metrics.run_declarations(PILOT / r)])
decls = decls.join(runs[["topic", "model", "condition"]], on="run")

fig, _, panels = cond_grid("declared position y", "share")
for ax, cond, model, topic in panels:
    y = decls[(decls.model == model) & (decls.topic == topic) & (decls.condition == cond)].y
    if len(y):
        ax.bar(POS, y.value_counts(normalize=True).reindex(POS, fill_value=0), color=COLOR, width=0.7)
    ax.set_xticks(POS)
    ax.set_ylim(0, 1)
    ax.grid(axis="y", color="0.92")
    ax.spines[["top", "right"]].set_visible(False)
    n_label(ax, len(y))
fig.suptitle("Distribution of the declared positions, pooled over p_inter", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 6. Change of the declaration against the memory

Whether a declaration moves toward its memory beyond persistence: every declaration of § 5 that
has a predecessor (the agent's previous modelled declaration y′; the first, which follows the start, has
none) and a window, its change Δy = y − y′ (0, or ±2 on a switch) against the memory's distance from
y′, the window's mean ŝ (first figure) or mean ô (second) minus y′, rounded to the nearest integer, a
tie toward 0. Column d shades the share of declarations at each Δy; each column sums to 1, white
where no distance falls. The dotted diagonal is full adoption (Δy = d), the dashed horizontal no move
(Δy = 0). ô itself is never shown: `full` shows the partner's turn it reads, `own` not even that; n the
declarations shown. A column holds as many declarations as fall there,
so a thin one is noisy. Pooled over each cell's complete runs and p_inter, on the grid of § 2. A
descriptive look, not a test.

In [ ]:
import numpy as np

STEPS = list(range(-2, 3))  # every rounded distance of a mean in [−1, 1] from a pole


def nearest(m):
    """The nearest integer, a tie toward 0, so neither direction gains it."""
    return (np.sign(m) * np.ceil(np.abs(m) - 0.5)).astype(int)


moves = decls.assign(prev=decls.groupby(["run", "agent"]).y.shift())  # run_declarations is in event order
moves = moves.dropna(subset=["prev"]).assign(dy=lambda t: (t.y - t.prev).astype(int))

for read, col in (("ŝ", "s_hat"), ("ô", "o_hat")):
    fig, grid, panels = cond_grid(f"mean {read} − y′", "Δy", figsize=(16, 8))
    for ax, cond, model, topic in panels:
        t = moves[(moves.model == model) & (moves.topic == topic) & (moves.condition == cond)]
        t = t.dropna(subset=[col])
        if len(t):
            share = (pd.crosstab(t.dy, nearest(t[col] - t.prev), normalize="columns")
                     .reindex(index=STEPS, columns=STEPS, fill_value=0))  # rows: Δy, columns: distance
            ax.imshow(share, origin="lower", cmap="Blues", vmin=0, vmax=1, extent=(-2.5, 2.5, -2.5, 2.5))
        ax.plot([-2.5, 2.5], [-2.5, 2.5], color="0.5", ls=":", lw=0.8)
        ax.axhline(0, color="0.5", ls="--", lw=0.8)
        ax.set_xticks(STEPS)
        ax.set_yticks(STEPS)
        n_label(ax, len(t))
    fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(0, 1), "Blues"), ax=grid, shrink=0.6,
                 label="share of declarations given the distance")
    fig.suptitle(f"Change of the declaration against the mean {read} in memory, pooled over p_inter",
                 fontsize=11, y=1.0)
    plt.show()

## 7. Response curve

§ 6 as one number per column: the mean change Δy at each distance of the memory from y′
(mean ŝ, first figure, or mean ô, second, minus y′, rounded as in § 6), `full` and `own`
in one panel, by model (rows) and topic (columns). A flat curve on the dashed line is no response; a
slope of 1 would be full adoption. `own` shows no partner, so its ô curve should stay flat; the gap between the two is what the partner's turns add. A
hollow point rests on fewer than `metrics.THIN_N` declarations; n gives the declarations under
`full` / `own`. A descriptive look, not a test.

In [ ]:
COND_COLOR = {"full": COLOR, "own": "#d9822b"}

for read, col in (("ŝ", "s_hat"), ("ô", "o_hat")):
    fig, axes = plt.subplots(len(MODELS), len(TOPICS), figsize=(8.5, 6), sharex=True, sharey=True,
                             squeeze=False)
    for (i, model), (j, topic) in itertools.product(enumerate(MODELS), enumerate(TOPICS)):
        ax = axes[i, j]
        ns = []
        for cond in CONDS:
            t = moves[(moves.model == model) & (moves.topic == topic) & (moves.condition == cond)]
            t = t.dropna(subset=[col])
            ns.append(len(t))
            g = t.dy.groupby(nearest(t[col] - t.prev)).agg(["mean", "size"]).reindex(STEPS)
            thick = g["size"] >= metrics.THIN_N
            ax.plot(g.index, g["mean"], color=COND_COLOR[cond], lw=1.4)
            ax.plot(g.index[thick], g["mean"][thick], "o", color=COND_COLOR[cond], ms=3.5)
            ax.plot(g.index[~thick], g["mean"][~thick], "o", mfc="white", mec=COND_COLOR[cond], ms=3.5)
        ax.axhline(0, color="0.5", ls="--", lw=0.8)
        ax.set_xticks(STEPS)
        ax.grid(axis="y", color="0.92")
        ax.spines[["top", "right"]].set_visible(False)
        n_label(ax, " / ".join(map(str, ns)) if any(ns) else 0)
        if i == 0:
            ax.set_title(topic, fontsize=10, pad=14)
        if j == 0:
            ax.set_ylabel(f"{model.split('-')[0]}\nmean Δy", fontsize=9)
        if i == len(MODELS) - 1:
            ax.set_xlabel(f"mean {read} − y′")
    fig.legend(handles=[Line2D([], [], color=COND_COLOR[c], lw=1.4, marker="o", ms=3.5, label=c) for c in CONDS]
                       + [Line2D([], [], color="0.4", ls="", marker="o", mfc="white", ms=3.5,
                                 label=f"n < {metrics.THIN_N}")],
               loc="center right", frameon=False, fontsize=8)
    fig.suptitle(f"Response of the declaration to the mean {read} in memory, pooled over p_inter", fontsize=11)
    fig.tight_layout(rect=(0, 0, 0.88, 0.97))
    plt.show()

### Reward Distributions

In [ ]:
# each run's mean r̂ = ŝ·ô over its turns per round, both sides of every exchange
reward = {r: metrics.run_turns(PILOT / r).groupby("round").r_hat.mean() for r in runs.index[runs.complete]}

plot_grid(lambda r: [(reward[r], "-")], "mean r̂", "Mean perceived reward r̂ per round", (-1.05, 1.05))